In [154]:
from github import Github

# Authentication is defined via github.Auth
from github import Auth

# using an access token
auth = Auth.Token(None)

# First create a Github instance:

# Public Web Github
g = Github(auth=auth)

# # Github Enterprise with custom hostname
# g = Github(base_url="https://{hostname}/api/v3", auth=auth)

In [155]:
repo = g.get_repo("PyGithub/PyGithub")
repo

Repository(full_name="PyGithub/PyGithub")

In [156]:
import os

from dotenv import load_dotenv
from github import Github, Auth, GithubException

load_dotenv()

github_api_key = os.getenv("GITHUB_API_KEY")
auth = Auth.Token(github_api_key)
g = Github(auth=auth)


class GithubContentTraverser:

    def get_metadata(content):
        metadata = content._rawData
        return metadata

    @classmethod
    def traverse(cls, repo, path:str="", cur_depth:int=0, max_depth:int=10):
        file_names = []

        if cur_depth >= max_depth:
            return file_names

        contents = repo.get_contents(path)
        if not isinstance(contents, list):
            contents = [contents]
        # return contents

        for content in contents:
            if content.type == "dir":
                file_names.extend(cls.traverse(repo, content.path, cur_depth+1, max_depth))
            else:
                # file_names.append(content)
                file_names.append(cls.get_metadata(content))

        return file_names

class GithubAPITOOL:

    def __init__(self, client):
        self.client = client

    def get_repo(
        self,
        full_name_or_id: int | str,
        get_stargazers_count:bool = True,
        # get_issues:bool = True,
        get_contents:str | None = "",
        get_branch:str | None = None,
        get_commits:str | None = None,
    ):
        repo = self.client.get_repo(full_name_or_id=full_name_or_id)

        metadata = {}
        if get_stargazers_count:
            metadata['stargazers_count'] = repo.stargazers_count

        # if get_issues:
        #     issues = repo.get_issues(state='all')[:10]
        #     issues = [filter_attrs(issue, pattern='all') for issue in issues]
        #     metadata['issues'] = issues

        if isinstance(get_contents, str):
            path = get_contents
            contents = GithubContentTraverser.traverse(repo, path, max_depth=3)
            metadata['contents'] = contents

        if isinstance(get_branch, str):
            branch_name = get_branch
            try:
                branch_meta = repo.get_branch(branch_name)._rawData
            except GithubException as e:
                branch_meta = e._GithubException__data
            except:
                branch_meta = {
                    'message': 'Unknown',
                    'documentation_url': 'https://docs.github.com/rest/branches/branches#get-a-branch',
                    'status': '503'
                }
            metadata['branch'] = branch_meta

        if isinstance(get_commits, str):
            sha = get_commits
            commits = [commit_meta._rawData for commit_meta in repo.get_commits(sha)[:10]]
            metadata['commits'] = commits

        return metadata


In [157]:
github_tool = GithubAPITOOL(g)
response = github_tool.get_repo("BAAI-DCAI/Bunny", get_contents="README.md")
response

{'stargazers_count': 909,
 'contents': [{'name': 'README.md',
   'path': 'README.md',
   'sha': '13a1af1e969e4ece60faa0ae4af8f37774fc6c72',
   'size': 35704,
   'url': 'https://api.github.com/repos/BAAI-DCAI/Bunny/contents/README.md?ref=main',
   'html_url': 'https://github.com/BAAI-DCAI/Bunny/blob/main/README.md',
   'git_url': 'https://api.github.com/repos/BAAI-DCAI/Bunny/git/blobs/13a1af1e969e4ece60faa0ae4af8f37774fc6c72',
   'download_url': 'https://raw.githubusercontent.com/BAAI-DCAI/Bunny/main/README.md',
   'type': 'file',
   'content': '...\nIFZpc2lvbiBBc3Npc3RhbnQuCg==\n',
   'encoding': 'base64',
   '_links': {'self': 'https://api.github.com/repos/BAAI-DCAI/Bunny/contents/README.md?ref=main',
    'git': 'https://api.github.com/repos/BAAI-DCAI/Bunny/git/blobs/13a1af1e969e4ece60faa0ae4af8f37774fc6c72',
    'html': 'https://github.com/BAAI-DCAI/Bunny/blob/main/README.md'}}]}

In [149]:
repo = g.get_repo("PyGithub/PyGithub")
commits = list(repo.get_commits()[:5])
commits[0]._rawData

{'sha': 'c44ec523b1830c2d747fcf417b478562854664e8',
 'node_id': 'C_kwDOADYVqtoAKGM0NGVjNTIzYjE4MzBjMmQ3NDdmY2Y0MTdiNDc4NTYyODU0NjY0ZTg',
 'commit': {'author': {'name': 'Min RK',
   'email': 'benjaminrk@gmail.com',
   'date': '2024-10-09T15:17:08Z'},
  'committer': {'name': 'GitHub',
   'email': 'noreply@github.com',
   'date': '2024-10-09T15:17:08Z'},
  'message': 'Make requester a public attribute (#3056)\n\nThere will always be new APIs not yet supported by PyGitHub (e.g.\n#2718). This adds an escape hatch for users to send requests directly\nwithout having to start from scratch or use private APIs while waiting\nfor PyGitHub to add support.\n\nChanges: \n\n- private `object._requester` `.__requester` is now available via public\n`.requester` attribute on MainClass, GitHubObject, etc.\n- add Requester APIs to docs as public methods, indicating status as\nstable public APIs\n\nFixes #2071 which received only positive feedback, but was closed for\ninactivity.\n\n---------\n\nCo-authore